In [1]:
import pandas as pd
import numpy as np
import glob
import os
from minicons import cwe
import torch
from tqdm import tqdm
from sklearn.cluster import KMeans

/home/gsc685/.conda/envs/acl_metapragmatics/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
Load the Features
"""


#we have 30 pretty common but not too highly polysemous (<15 senses) words and all of their okens from semcor
# at least 50 tokens each, I think. 

# the data for each word are stored in different files in ./features/semcor. 
# we want to load them all into a single dataframe.

# Set your directory path
folder_path = "/home/gsc685/data/features/semcor/"
# Get all CSV file paths in the directory
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
print(csv_files)

# Read and concatenate all CSV files into a single DataFrame
all_data = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Preview
print(all_data.head())

['/home/gsc685/data/features/semcor/left_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/obtained_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/same_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/general_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/built_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/door_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/one_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/high_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/third_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/information_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/first_feature_vectors_roberta_buchanan_layer7.csv', '/home/gsc685/data/features/semcor/di

In [3]:
"""
Load the Sentences and calculate embeddings
"""

tokens = pd.read_csv("/home/gsc685/data/collected_tokens/semcor/semcor_validation_tokens.csv")

# it might be that we didnt get data for some sentences so ... lets exclude them. 
indexes_to_keep = all_data['token_id'].unique().tolist()
print(len(tokens))
tokens = tokens[tokens["sentence_id"].isin(indexes_to_keep)]
print(len(tokens))

# join with corpus file that has sentences
corpus = pd.read_csv('/home/gsc685/data/semcor_corpus.csv')
tokens = tokens.merge(corpus, how="left", left_on="sentence_id", right_on="id")

tokens.head()



3858
3577


,Unnamed: 0,lemma,sense,word_form,sentence_id,pos,sentence,id
0,0,left,Lemma('leave.v.03.leave'),left,25193,VB,Yet your list of things left undone did not in...,25193
1,1,obtained,Lemma('obtain.v.01.obtain'),obtained,32325,VB,It embraced determining when to purchase and w...,32325
2,2,same,Lemma('same.a.01.same'),same,2158,JJ,We know that the number of radio and televisio...,2158
3,3,same,Lemma('same.a.02.same'),same,7738,JJ,As Kate came swiftly down the stairs to the ha...,7738
4,5,built,Lemma('construct.v.01.build'),built,28702,VB,A new waterfront site for the bureau is now be...,28702


In [4]:
# split the data in half with an evenish number of tokens for each sense on both sides
# Let's say you want to split based on group column 'group'
group_col = 'sense'  # Replace with your actual column name

# Split each group roughly in half
grouped = tokens.groupby(group_col)

df1_list = []
df2_list = []

for _, group in grouped:
    group = group.sample(frac=1, random_state=42)  # Shuffle within group
    half = len(group) // 2
    df1_list.append(group.iloc[:half])
    df2_list.append(group.iloc[half:])

# Combine back into two DataFrames
df1 = pd.concat(df1_list).reset_index(drop=True)
df2 = pd.concat(df2_list).reset_index(drop=True)

In [5]:
print(len(df1))
print(len(df2))

1721
1856


In [6]:
df1.head()

,Unnamed: 0,lemma,sense,word_form,sentence_id,pos,sentence,id
0,831,took,Lemma('accept.v.02.take'),took,19173,VB,"She took it grudgingly , her dark eyes baleful...",19173
1,3657,took,Lemma('accept.v.02.take'),took,14290,VB,He took the story of the pound of flesh and ha...,14290
2,83,age,Lemma('age.n.01.age'),age,3889,NN,The differences between onset age and completi...,3889
3,268,age,Lemma('age.n.01.age'),age,3915,NN,"The 34 arrows , denoting onset age plus comple...",3915
4,2396,age,Lemma('age.n.01.age'),age,3713,NN,Carpenter 's study showed that female common g...,3713


In [7]:
# now you need roberta embeddings for the sentences in each half.
_device = "cuda:1" if torch.cuda.is_available() else "cpu"
_embedding_model = cwe.CWE('roberta-base', device = _device)
_batch_size=75

# helper function to batch process inputs
def batch_iterable(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i + batch_size]


def get_embeddings(df):

    # data as list of tuples
    # data = list(zip(sentences, word))

    # save all queries separately
    # (needed because some words do not occur in
    # sentences in the same form and must be fixed first)
    words = df['word_form']
    sentences = df['sentence']


    queries = list(zip(sentences, words))
    embs = []
    for i, batch in tqdm(enumerate(batch_iterable(queries, _batch_size))):

        batch_embs = _embedding_model.extract_representation(batch, layer=7).cpu().detach().numpy()
        embs.append(batch_embs)

    return np.vstack(embs)

df1_embs = get_embeddings(df1)
df2_embs = get_embeddings(df2)

/home/gsc685/.conda/envs/acl_metapragmatics/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
23it [00:04,  4.78it/s]
25it [00:04,  6.18it/s]


In [8]:
"""
clusterize - but according to the whole dataset. 
"""

# first -- how many unique lemmas are there?
print(len(tokens["lemma"].unique()))

# there are 30 so we'll need 30 clusters
k_means_n = 30

def cluster(embeddings):
    """
    input: list of roberta embeddings of a single layer
    output: list of cluster IDs. 
    """
    kmeans_obj = KMeans(n_clusters=k_means_n, n_init=10)
    kmeans_obj.fit(embeddings)

    #label_list = kmeans_obj.labels_
    #cluster_centroids = kmeans_obj.cluster_centers_

    clusters = kmeans_obj.fit_predict(embeddings)
    return clusters

df1_clusters = cluster(df1_embs)
df2_clusters = cluster(df2_embs)

30


In [9]:
df1_clusters

array([ 4,  4, 19, ..., 21, 21, 21], dtype=int32)

In [10]:
# add the cluster info to the tokens dataframes
df1['cluster'] = df1_clusters
df2['cluster'] = df2_clusters

df1.head()

,Unnamed: 0,lemma,sense,word_form,sentence_id,pos,sentence,id,cluster
0,831,took,Lemma('accept.v.02.take'),took,19173,VB,"She took it grudgingly , her dark eyes baleful...",19173,4
1,3657,took,Lemma('accept.v.02.take'),took,14290,VB,He took the story of the pound of flesh and ha...,14290,4
2,83,age,Lemma('age.n.01.age'),age,3889,NN,The differences between onset age and completi...,3889,19
3,268,age,Lemma('age.n.01.age'),age,3915,NN,"The 34 arrows , denoting onset age plus comple...",3915,19
4,2396,age,Lemma('age.n.01.age'),age,3713,NN,Carpenter 's study showed that female common g...,3713,19


In [11]:
# save these lists with clusters to disk

df1.to_csv('/home/gsc685/data/validation_tokens_semcor_left.csv')
df2.to_csv('/home/gsc685/data/validation_tokens_semcor_right.csv')


# not doing anything below this anymore

In [12]:
# look at the overlap of grouping into lemmas and clusters
df1.groupby(['cluster', 'lemma']).count()

Unnamed: 0  sense  word_form  sentence_id  pos  sentence  \
cluster lemma                                                                   
0       obtained             45     45         45           45   45        45   
1       first                44     44         44           44   44        44   
2       one                  36     36         36           36   36        36   
3       study                64     64         64           64   64        64   
4       took                 84     84         84           84   84        84   
5       same                105    105        105          105  105       105   
6       clear                26     26         26           26   26        26   
        suppose               7      7          7            7    7         7   
7       growth               27     27         27           27   27        27   
8       seem                 72     72         72           72   72        72   
        suppose              13     13         13           13   13        13   
9       considered           41     41         41           41   41        41   
10      high                 67     67         67           67   67        67   
11      door                 65     65         65           65   65        65   
12      left                 37     37         37           37   37        37   
13      large                56     56         56           56   56        56   
14      first               129    129        129          129  129       129   
15      animal               24     24         24           24   24        24   
        government           30     30         30           30   30        30   
16      no                   62     62         62           62   62        62   
17      general              38     38         38           38   38        38   
18      ready                27     27         27           27   27        27   
19      age                  41     41         41           41   41        41   
20      third                40     40         40           40   40        40   
21      word                 74     74         74           74   74        74   
22      building             30     30         30           30   30        30   
        built                31     31         31           31   31        31   
23      meaning              32     32         32           32   32        32   
24      died                 29     29         29           29   29        29   
25      left                 21     21         21           21   21        21   
26      information          65     65         65           65   65        65   
27      no                    1      1          1            1    1         1   
        ran                  23     23         23           23   23        23   
        stand                41     41         41           41   41        41   
28      left                 49     49         49           49   49        49   
29      one                 145    145        145          145  145       145   

                      id  
cluster lemma             
0       obtained      45  
1       first         44  
2       one           36  
3       study         64  
4       took          84  
5       same         105  
6       clear         26  
        suppose        7  
7       growth        27  
8       seem          72  
        suppose       13  
9       considered    41  
10      high          67  
11      door          65  
12      left          37  
13      large         56  
14      first        129  
15      animal        24  
        government    30  
16      no            62  
17      general       38  
18      ready         27  
19      age           41  
20      third         40  
21      word          74  
22      building      30  
        built         31  
23      meaning       32  
24      died          29  
25      left          21  
26      information   65  
27      no             1  
        ran           23

In [13]:
"""
So building_n and built_v are assigned the same cluster and another word got split into two. This is to be expected as we are dealing with polysemy. But overall its a pretty sharp split.

Now, I want to get average or distinctive feature values for each cluster. 
    - inputs: features, tokens w/ clusters
    - outputs: df of n_clusters x n_features


"""

# get separate features
df1_indexes = df1["sentence_id"].tolist()
features1 = all_data[all_data['token_id'].isin(df1_indexes)]
print(len(features1) /3981)

print(len(all_data)/3981)
merged = df1.merge(all_data, 
                       right_on='token_id',
                       left_on='sentence_id' ,
                       how='left')

print(len(df1))
print(len(merged)/3981)



1947.0
3577.0
1721
2193.0


In [14]:
# Group by 'cluster' and 'feature' and compute the mean
grouped = merged.groupby(['feature', 'cluster'])['predicted_value'].mean().reset_index()

# Pivot so that each cluster is a column
pivoted = grouped.pivot(index='feature', columns='cluster', values='predicted_value')

# Optional: reset column names if needed
pivoted.columns.name = None  # removes the 'cluster' label above the columns

# See the result
print(pivoted.head())

KeyError: 'cluster'